# RAG Evaluation: Exception Retrieval Without Narrative VS With Narrative

This notebook evaluates RAG retrieval quality by comparing raw exception data against narrative-formatted exception data.

## 1. Import Required Libraries

In [1]:
import pandas as pd
import numpy as np
import json
import httpx
from typing import List, Dict, Any
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer
import logging
import nest_asyncio

# Allow nested event loops in Jupyter
nest_asyncio.apply()

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

## 2. Connect to Milvus and Fetch All Documents

In [2]:
from pymilvus import connections, Collection

# Connect to Milvus on port 19530
connections.connect(
    alias="default",
    host="localhost",
    port=19530
)

# Load the 'documents' collection
collection = Collection("documents")

# Fetch all documents - only get text and metadata fields
results = collection.query(
    expr="",  # Empty expression to get all documents
    output_fields=["text", "metadata"],
    limit=1000,  # Set a high limit to fetch all documents
    consistency_level="Strong"
)

# Parse the documents and metadata
documents_list = []
for doc in results:
    # Parse metadata JSON string if it's a string
    metadata = doc.get("metadata", {})
    if isinstance(metadata, str):
        metadata = json.loads(metadata)
    
    # Extract exception_id from metadata
    exception_id = metadata.get("exception_id")
    
    documents_list.append({
        "text": doc.get("text", ""),
        "metadata": metadata,
        "exception_id": exception_id,
        "trade_id": metadata.get("trade_id"),
        "priority": metadata.get("priority"),
        "status": metadata.get("trade_status"),
        "asset_type": metadata.get("asset_type"),
        "clearing_house": metadata.get("clearing_house"),
        "exception_msg": metadata.get("exception_msg"),
        "problem_category": metadata.get("problem_category")
    })

# Convert to DataFrame
documents_df = pd.DataFrame(documents_list)
print(f"Total documents fetched: {len(documents_df)}")
print(f"\nDataFrame shape: {documents_df.shape}")
print(f"\nFirst few rows:")
print(documents_df[['exception_id', 'trade_id', 'priority', 'asset_type', 'exception_msg']].head())
print(f"\nDocument columns: {documents_df.columns.tolist()}")
print(f"\nSample text (first 500 chars):")
print(documents_df.iloc[0]['text'][:500] if len(documents_df) > 0 else "No documents")

Total documents fetched: 43

DataFrame shape: (43, 10)

First few rows:
  exception_id  trade_id  priority asset_type        exception_msg
0     19757932  55623053      HIGH        IRS  INSUFFICIENT MARGIN
1     86572959  68186799      HIGH         FX        MAPPING ISSUE
2     20001001  77194044  CRITICAL        IRS    TIME OUT OF RANGE
3     20002001  60724962      HIGH        IRS  INSUFFICIENT MARGIN
4     20003001  86836834      HIGH        CDS        MAPPING ISSUE

Document columns: ['text', 'metadata', 'exception_id', 'trade_id', 'priority', 'status', 'asset_type', 'clearing_house', 'exception_msg', 'problem_category']

Sample text (first 500 chars):
{
  "exception": {
    "trade_id": 55623053,
    "trans_id": 88242387,
    "msg": "INSUFFICIENT MARGIN",
    "priority": "HIGH",
    "status": "CLOSED",
    "comment": "RETRY LIMIT EXCEEDED",
    "id": 19757932,
    "create_time": "2025-08-07T09:12:25Z",
    "update_time": "2025-08-07T09:12:25Z"
  },
  "trade": {
    "id": 55623053,


In [3]:
documents_df

,text,metadata,exception_id,trade_id,priority,status,asset_type,clearing_house,exception_msg,problem_category
0,"{\n ""exception"": {\n ""trade_id"": 55623053,...","{'trade_id': '55623053', 'exception_id': '1975...",19757932,55623053,HIGH,ALLEGED,IRS,JSCC,INSUFFICIENT MARGIN,Collateral and Margin Issue
1,"{\n ""exception"": {\n ""trade_id"": 68186799,...","{'trade_id': '68186799', 'exception_id': '8657...",86572959,68186799,HIGH,ALLEGED,FX,CME,MAPPING ISSUE,Legal Documentation Gap
2,"{\n ""exception"": {\n ""trade_id"": 77194044,...","{'trade_id': '77194044', 'exception_id': '2000...",20001001,77194044,CRITICAL,REJECTED,IRS,LCH,TIME OUT OF RANGE,Collateral and Margin Issue
3,"{\n ""exception"": {\n ""trade_id"": 60724962,...","{'trade_id': '60724962', 'exception_id': '2000...",20002001,60724962,HIGH,REJECTED,IRS,OTCCHK,INSUFFICIENT MARGIN,Collateral and Margin Issue
4,"{\n ""exception"": {\n ""trade_id"": 86836834,...","{'trade_id': '86836834', 'exception_id': '2000...",20003001,86836834,HIGH,REJECTED,CDS,JSCC,MAPPING ISSUE,Collateral and Margin Issue | Legal Documentat...
5,"{\n ""exception"": {\n ""trade_id"": 77186642,...","{'trade_id': '77186642', 'exception_id': '2000...",20004001,77186642,MEDIUM,REJECTED,CDS,LCH,MISSING BIC,Settlement Instructions Problem
6,"{\n ""exception"": {\n ""trade_id"": 61270683,...","{'trade_id': '61270683', 'exception_id': '2000...",20005001,61270683,MEDIUM,REJECTED,FX,LCH,TIME OUT OF RANGE,Collateral and Margin Issue
7,"{\n ""exception"": {\n ""trade_id"": 98962729,...","{'trade_id': '98962729', 'exception_id': '2000...",20006001,98962729,CRITICAL,REJECTED,CDS,JSCC,TIME OUT OF RANGE,Collateral and Margin Issue
8,"{\n ""exception"": {\n ""trade_id"": 25399440,...","{'trade_id': '25399440', 'exception_id': '2000...",20007001,25399440,HIGH,REJECTED,IRS,LCH,MAPPING ISSUE,Legal Documentation Gap
9,"{\n ""exception"": {\n ""trade_id"": 53850480,...","{'trade_id': '53850480', 'exception_id': '2000...",20008001,53850480,MEDIUM,REJECTED,CDS,LCH,INSUFFICIENT MARGIN,Collateral and Margin Issue


## 3. Define Test Exception IDs

In [5]:
# Test exception IDs (convert to strings to match dataframe format)
test_exception_ids = ["20038001", "20039001", "20040001"]

# Verify these exceptions exist in our dataset
print("Test Exception IDs:")
print(f"Sample exception_id type in dataframe: {type(documents_df.iloc[0]['exception_id'])}")
for exc_id in test_exception_ids:
    exc_data = documents_df[documents_df['exception_id'] == exc_id]
    if len(exc_data) > 0:
        print(f"  {exc_id}: Found ✓")
    else:
        print(f"  {exc_id}: NOT FOUND ✗")

Test Exception IDs:
Sample exception_id type in dataframe: <class 'str'>
  20038001: Found ✓
  20039001: Found ✓
  20040001: Found ✓


## 4. Manual Similarity Analysis (Using Text Data Only)

For each test exception, identify the 3 most similar exceptions based on text similarity using TF-IDF vectorizer.

In [6]:
def find_manual_similar_exceptions(query_exception_id: int, documents_df: pd.DataFrame, k: int = 3) -> List[Dict[str, Any]]:
    """
    Manually find k most similar exceptions by analyzing metadata and text patterns.
    Uses judgment-based similarity scoring based on key attributes.
    """
    # Get the query exception
    query_doc = documents_df[documents_df['exception_id'] == query_exception_id]
    if len(query_doc) == 0:
        print(f"Exception {query_exception_id} not found")
        return []
    
    query = query_doc.iloc[0]
    query_text = query['text']
    query_metadata = query['metadata']
    
    print(f"\n{'='*80}")
    print(f"Looking for similar exceptions to: {query_exception_id}")
    print(f"{'='*80}")
    print(f"Query Exception Details:")
    print(f"  - Asset Type: {query['asset_type']}")
    print(f"  - Clearing House: {query['clearing_house']}")
    print(f"  - Priority: {query['priority']}")
    print(f"  - Status: {query['status']}")
    print(f"  - Exception Message: {query['exception_msg']}")
    print(f"  - Problem Category: {query['problem_category']}")
    print(f"  - Trade ID: {query['trade_id']}")
    
    # Calculate similarity scores for each other exception
    similarity_scores = []
    
    for idx, row in documents_df.iterrows():
        if row['exception_id'] == query_exception_id:
            continue
        
        # Start with base similarity score
        score = 0.0
        
        # Exact matches (strong indicators of similarity)
        if row['asset_type'] == query['asset_type']:
            score += 0.25
        if row['clearing_house'] == query['clearing_house']:
            score += 0.25
        if row['priority'] == query['priority']:
            score += 0.15
        if row['problem_category'] == query['problem_category']:
            score += 0.15
        
        # Partial matches (moderate indicators)
        if row['status'] == query['status']:
            score += 0.10
        
        # Text similarity (simple substring matching)
        query_text_lower = query_text.lower()
        row_text_lower = row['text'].lower()
        
        # Check for common keywords in exception messages
        if row['exception_msg'] and query['exception_msg']:
            if row['exception_msg'].lower() in query_text_lower or query['exception_msg'].lower() in row_text_lower:
                score += 0.10
        
        # Check if trade IDs appear in each other's texts (trades might be related)
        if str(query['trade_id']) in row_text_lower:
            score += 0.05
        if str(row['trade_id']) in query_text_lower:
            score += 0.05
        
        similarity_scores.append({
            'exception_id': row['exception_id'],
            'score': score,
            'asset_type': row['asset_type'],
            'clearing_house': row['clearing_house'],
            'priority': row['priority'],
            'problem_category': row['problem_category']
        })
    
    # Sort by score descending and get top k
    similarity_scores.sort(key=lambda x: x['score'], reverse=True)
    top_k = similarity_scores[:k]
    
    manual_results = []
    for rank, result in enumerate(top_k, 1):
        manual_results.append({
            'exception_id': result['exception_id'],
            'similarity_score': result['score'],
            'rank': rank,
            'reasoning': f"Asset: {result['asset_type']}, House: {result['clearing_house']}, Priority: {result['priority']}"
        })
        print(f"\n  Rank {rank}: Exception {result['exception_id']}")
        print(f"    - Similarity Score: {result['score']:.2f}")
        print(f"    - Asset Type: {result['asset_type']}")
        print(f"    - Clearing House: {result['clearing_house']}")
        print(f"    - Priority: {result['priority']}")
        print(f"    - Problem Category: {result['problem_category']}")
    
    return manual_results

# Run manual similarity analysis for each test exception
manual_results = {}
for test_exc_id in test_exception_ids:
    manual_results[test_exc_id] = find_manual_similar_exceptions(test_exc_id, documents_df, k=3)

print(f"\n{'='*80}")
print("Manual Similarity Analysis Complete")
print(f"{'='*80}")


Looking for similar exceptions to: 20038001
Query Exception Details:
  - Asset Type: CDS
  - Clearing House: LCH
  - Priority: CRITICAL
  - Status: CANCELLED
  - Exception Message: INSUFFICIENT MARGIN
  - Problem Category: Collateral and Margin Issue
  - Trade ID: 58392014

  Rank 1: Exception 20036001
    - Similarity Score: 0.85
    - Asset Type: CDS
    - Clearing House: LCH
    - Priority: HIGH
    - Problem Category: Collateral and Margin Issue

  Rank 2: Exception 20008001
    - Similarity Score: 0.75
    - Asset Type: CDS
    - Clearing House: LCH
    - Priority: MEDIUM
    - Problem Category: Collateral and Margin Issue

  Rank 3: Exception 20009001
    - Similarity Score: 0.75
    - Asset Type: CDS
    - Clearing House: LCH
    - Priority: HIGH
    - Problem Category: Collateral and Margin Issue

Looking for similar exceptions to: 20039001
Query Exception Details:
  - Asset Type: FX
  - Clearing House: CME
  - Priority: MEDIUM
  - Status: CANCELLED
  - Exception Message: MAPP

## 5. RAG Evaluation: Raw JSON Data

### 5.1 Query RAG Endpoint (Raw JSON)

Query the RAG service using the original raw JSON exception data to get similar exceptions.

In [11]:
async def get_rag_similar_exceptions(exception_id: str, limit: int = 3) -> List[Dict[str, Any]]:
    """
    Query the RAG service endpoint to get similar exceptions.
    """
    async with httpx.AsyncClient(base_url="http://localhost:8004", timeout=30.0) as client:
        try:
            response = await client.get(
                f"/api/rag/documents/similar-exceptions/{exception_id}",
                params={"limit": limit, "explain": False}
            )
            if response.status_code == 200:
                data = response.json()
                similar_exceptions = data.get('similar_exceptions', [])
                
                rag_results = []
                for rank, exc in enumerate(similar_exceptions, 1):
                    rag_results.append({
                        'exception_id': exc['exception_id'],
                        'similarity_score': exc.get('similarity_score', 0),
                        'rank': rank,
                        'priority': exc.get('priority'),
                        'status': exc.get('status')
                    })
                return rag_results
            else:
                logger.error(f"RAG endpoint returned status {response.status_code}: {response.text}")
                return []
        except Exception as e:
            logger.error(f"Error calling RAG endpoint: {str(e)}")
            return []

# Run RAG queries for each test exception using top-level await
rag_results = {}
for test_exc_id in test_exception_ids:
    rag_results[test_exc_id] = await get_rag_similar_exceptions(test_exc_id, limit=3)
    print(f"\nRAG Top 3 Similar Exceptions for {test_exc_id}:")
    if rag_results[test_exc_id]:
        for result in rag_results[test_exc_id]:
            print(f"  Rank {result['rank']}: Exception {result['exception_id']} (similarity: {result['similarity_score']:.4f})")
    else:
        print(f"  No results returned from RAG")

INFO:httpx:HTTP Request: GET http://localhost:8004/api/rag/documents/similar-exceptions/20038001?limit=3&explain=false "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET http://localhost:8004/api/rag/documents/similar-exceptions/20039001?limit=3&explain=false "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET http://localhost:8004/api/rag/documents/similar-exceptions/20040001?limit=3&explain=false "HTTP/1.1 200 OK"



RAG Top 3 Similar Exceptions for 20038001:
  Rank 1: Exception 20040001 (similarity: 56.5100)
  Rank 2: Exception 20039001 (similarity: 53.9700)
  Rank 3: Exception 20035001 (similarity: 49.7400)

RAG Top 3 Similar Exceptions for 20039001:
  Rank 1: Exception 20040001 (similarity: 74.6400)
  Rank 2: Exception 20024001 (similarity: 55.4500)
  Rank 3: Exception 20021001 (similarity: 54.7200)

RAG Top 3 Similar Exceptions for 20040001:
  Rank 1: Exception 20039001 (similarity: 74.6400)
  Rank 2: Exception 20038001 (similarity: 56.5100)
  Rank 3: Exception 20030001 (similarity: 55.7400)


### 5.2 Calculate Metrics and Assessment (Raw JSON)

Evaluate RAG retrieval quality using Precision@K and Context Precision metrics.

In [12]:
def calculate_precision_at_k(manual_results: List[Dict], rag_results: List[Dict], k: int = 3) -> float:
    """
    Calculate Precision@K: proportion of top-k RAG results that are relevant (in manual top-k).
    Relevant = exception IDs match between manual and RAG top-k.
    """
    manual_ids = set([r['exception_id'] for r in manual_results[:k]])
    rag_ids = [r['exception_id'] for r in rag_results[:k]]
    
    if len(rag_ids) == 0:
        return 0.0
    
    # Count how many of the RAG results are in the manual top-k
    relevant_count = sum(1 for rag_id in rag_ids if rag_id in manual_ids)
    precision = relevant_count / len(rag_ids)
    
    return precision

def calculate_context_precision(manual_results: List[Dict], rag_results: List[Dict], k: int = 3) -> float:
    """
    Calculate Context Precision: proportion of manual (ground truth) relevant exceptions
    that appear in the top-k RAG results.
    Measures: "Of the exceptions the domain expert identified as similar, 
    how many did the RAG system find in the top-k?"
    """
    manual_ids = set([r['exception_id'] for r in manual_results[:k]])
    rag_ids = set([r['exception_id'] for r in rag_results[:k]])
    
    if len(manual_ids) == 0:
        return 0.0
    
    # Count how many manual results appear in RAG top-k
    found_count = len(manual_ids.intersection(rag_ids))
    context_precision = found_count / len(manual_ids)
    
    return context_precision

# Generate metrics for each test exception
print("\n" + "="*70)
print("RAG RETRIEVAL EVALUATION METRICS (Precision@K & Context Precision)")
print("="*70)

metrics_data = []
for test_exc_id in test_exception_ids:
    precision_k = calculate_precision_at_k(manual_results[test_exc_id], rag_results[test_exc_id], k=3)
    context_prec = calculate_context_precision(manual_results[test_exc_id], rag_results[test_exc_id], k=3)
    
    manual_ids = [r['exception_id'] for r in manual_results[test_exc_id]]
    rag_ids = [r['exception_id'] for r in rag_results[test_exc_id]]
    
    metrics_data.append({
        'exception_id': test_exc_id,
        'precision_at_3': precision_k,
        'context_precision': context_prec,
        'manual_top_3': manual_ids,
        'rag_top_3': rag_ids
    })
    
    print(f"\n📌 Exception ID: {test_exc_id}")
    print(f"  Manual Top 3:        {manual_ids}")
    print(f"  RAG Top 3:           {rag_ids}")
    print(f"  Precision@3:         {precision_k:.2%}")
    print(f"  Context Precision:   {context_prec:.2%}")

# Create metrics DataFrame
metrics_df = pd.DataFrame(metrics_data)
print(f"\n\nMetrics Summary DataFrame:")
print(metrics_df[['exception_id', 'precision_at_3', 'context_precision']])


RAG RETRIEVAL EVALUATION METRICS (Precision@K & Context Precision)

📌 Exception ID: 20038001
  Manual Top 3:        ['20036001', '20008001', '20009001']
  RAG Top 3:           ['20040001', '20039001', '20035001']
  Precision@3:         0.00%
  Context Precision:   0.00%

📌 Exception ID: 20039001
  Manual Top 3:        ['20029001', '20024001', '86572959']
  RAG Top 3:           ['20040001', '20024001', '20021001']
  Precision@3:         33.33%
  Context Precision:   33.33%

📌 Exception ID: 20040001
  Manual Top 3:        ['20030001', '20033001', '20002001']
  RAG Top 3:           ['20039001', '20038001', '20030001']
  Precision@3:         33.33%
  Context Precision:   33.33%


Metrics Summary DataFrame:
  exception_id  precision_at_3  context_precision
0     20038001        0.000000           0.000000
1     20039001        0.333333           0.333333
2     20040001        0.333333           0.333333


## 6. RAG Evaluation: Narrative Data

### 6.1 Query RAG Endpoint (Narrative)

In [13]:
# Calculate aggregate metrics
avg_precision = metrics_df['precision_at_3'].mean()
avg_context_precision = metrics_df['context_precision'].mean()

print("\n" + "="*70)
print("AGGREGATE EVALUATION METRICS")
print("="*70)
print(f"\nTotal Test Exceptions: {len(test_exception_ids)}")
print(f"\nAverage Precision@3:        {avg_precision:.2%}")
print(f"Average Context Precision:  {avg_context_precision:.2%}")

# Interpretation
print("\n" + "-"*70)
print("Metric Definitions:")
print("-"*70)
print("\n• Precision@3: What fraction of RAG's top-3 results match manual (expert) judgments?")
print("  - Higher = RAG returns mostly relevant exceptions")
print("  - Formula: (# of RAG results matching manual top-3) / 3")

print("\n• Context Precision: What fraction of manual (expert) relevant exceptions")
print("  does RAG successfully retrieve in the top-3?")
print("  - Higher = RAG captures all the exceptions experts identified as similar")
print("  - Formula: (# of manual results found by RAG) / (# of manual results)")

print("\n" + "-"*70)
print("Quality Assessment:")
print("-"*70)

# Assessment based on both metrics
if avg_precision >= 0.8 and avg_context_precision >= 0.8:
    rating = "Excellent ✓✓"
elif avg_precision >= 0.6 or avg_context_precision >= 0.6:
    rating = "Good ✓"
elif avg_precision >= 0.4 or avg_context_precision >= 0.4:
    rating = "Fair ⚠️"
else:
    rating = "Poor ✗"

print(f"\nOverall RAG Retrieval Quality: {rating}")
print(f"\nInterpretation:")
if avg_precision >= 0.8:
    print("  ✓ Most of RAG's results are correct (high precision)")
else:
    print("  ✗ RAG includes many irrelevant results (low precision)")
    
if avg_context_precision >= 0.8:
    print("  ✓ RAG finds almost all relevant exceptions (high context precision)")
else:
    print("  ✗ RAG misses some relevant exceptions (low context precision)")


AGGREGATE EVALUATION METRICS

Total Test Exceptions: 3

Average Precision@3:        22.22%
Average Context Precision:  22.22%

----------------------------------------------------------------------
Metric Definitions:
----------------------------------------------------------------------

• Precision@3: What fraction of RAG's top-3 results match manual (expert) judgments?
  - Higher = RAG returns mostly relevant exceptions
  - Formula: (# of RAG results matching manual top-3) / 3

• Context Precision: What fraction of manual (expert) relevant exceptions
  does RAG successfully retrieve in the top-3?
  - Higher = RAG captures all the exceptions experts identified as similar
  - Formula: (# of manual results found by RAG) / (# of manual results)

----------------------------------------------------------------------
Quality Assessment:
----------------------------------------------------------------------

Overall RAG Retrieval Quality: Poor ✗

Interpretation:
  ✗ RAG includes many irre

### 6.2 Calculate Metrics and Assessment (Narrative)

Evaluate RAG retrieval quality with narrative data using the same metrics.


In [14]:
# Run RAG queries for narrative data using the same test exception IDs
# The Milvus database now contains narrative text instead of raw JSON
rag_results_narrative = {}
for test_exc_id in test_exception_ids:
    rag_results_narrative[test_exc_id] = await get_rag_similar_exceptions(test_exc_id, limit=3)
    print(f"\nRAG Top 3 Similar Exceptions for {test_exc_id} (Narrative Data):")
    if rag_results_narrative[test_exc_id]:
        for result in rag_results_narrative[test_exc_id]:
            print(f"  Rank {result['rank']}: Exception {result['exception_id']} (similarity: {result['similarity_score']:.4f})")
    else:
        print(f"  No results returned from RAG")

print("\n" + "="*70)
print("RAG Queries Complete (Narrative Data)")
print("="*70)


INFO:httpx:HTTP Request: GET http://localhost:8004/api/rag/documents/similar-exceptions/20038001?limit=3&explain=false "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET http://localhost:8004/api/rag/documents/similar-exceptions/20039001?limit=3&explain=false "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET http://localhost:8004/api/rag/documents/similar-exceptions/20040001?limit=3&explain=false "HTTP/1.1 200 OK"



RAG Top 3 Similar Exceptions for 20038001 (Narrative Data):
  Rank 1: Exception 20009001 (similarity: 91.8700)
  Rank 2: Exception 20008001 (similarity: 91.7800)
  Rank 3: Exception 20036001 (similarity: 91.2100)

RAG Top 3 Similar Exceptions for 20039001 (Narrative Data):
  Rank 1: Exception 20024001 (similarity: 88.5500)
  Rank 2: Exception 20021001 (similarity: 86.6100)
  Rank 3: Exception 20019001 (similarity: 83.6200)

RAG Top 3 Similar Exceptions for 20040001 (Narrative Data):
  Rank 1: Exception 20030001 (similarity: 87.1700)
  Rank 2: Exception 20025001 (similarity: 85.2000)
  Rank 3: Exception 20007001 (similarity: 84.9500)

RAG Queries Complete (Narrative Data)


## 7. Comparative Analysis: Raw JSON vs Narrative Data


In [15]:
# Generate metrics for narrative data - using same manual results and new RAG results
print("\n" + "="*70)
print("RAG RETRIEVAL EVALUATION METRICS - NARRATIVE DATA")
print("="*70)

metrics_data_narrative = []
for test_exc_id in test_exception_ids:
    precision_k = calculate_precision_at_k(manual_results[test_exc_id], rag_results_narrative[test_exc_id], k=3)
    context_prec = calculate_context_precision(manual_results[test_exc_id], rag_results_narrative[test_exc_id], k=3)
    
    manual_ids = [r['exception_id'] for r in manual_results[test_exc_id]]
    rag_ids = [r['exception_id'] for r in rag_results_narrative[test_exc_id]]
    
    metrics_data_narrative.append({
        'exception_id': test_exc_id,
        'precision_at_3': precision_k,
        'context_precision': context_prec,
        'manual_top_3': manual_ids,
        'rag_top_3': rag_ids
    })
    
    print(f"\n📌 Exception ID: {test_exc_id}")
    print(f"  Manual Top 3:        {manual_ids}")
    print(f"  RAG Top 3 (Narrative): {rag_ids}")
    print(f"  Precision@3:         {precision_k:.2%}")
    print(f"  Context Precision:   {context_prec:.2%}")

# Create metrics DataFrame for narrative
metrics_df_narrative = pd.DataFrame(metrics_data_narrative)
print(f"\n\nMetrics Summary DataFrame (Narrative Data):")
print(metrics_df_narrative[['exception_id', 'precision_at_3', 'context_precision']])

# Calculate aggregate metrics for narrative
avg_precision_narrative = metrics_df_narrative['precision_at_3'].mean()
avg_context_precision_narrative = metrics_df_narrative['context_precision'].mean()

print("\n" + "="*70)
print("AGGREGATE EVALUATION METRICS - NARRATIVE DATA")
print("="*70)
print(f"\nTotal Test Exceptions: {len(test_exception_ids)}")
print(f"\nAverage Precision@3:        {avg_precision_narrative:.2%}")
print(f"Average Context Precision:  {avg_context_precision_narrative:.2%}")

# Assessment for narrative data
if avg_precision_narrative >= 0.8 and avg_context_precision_narrative >= 0.8:
    rating_narrative = "Excellent ✓✓"
elif avg_precision_narrative >= 0.6 or avg_context_precision_narrative >= 0.6:
    rating_narrative = "Good ✓"
elif avg_precision_narrative >= 0.4 or avg_context_precision_narrative >= 0.4:
    rating_narrative = "Fair ⚠️"
else:
    rating_narrative = "Poor ✗"

print(f"\nRAG Retrieval Quality (Narrative): {rating_narrative}")



RAG RETRIEVAL EVALUATION METRICS - NARRATIVE DATA

📌 Exception ID: 20038001
  Manual Top 3:        ['20036001', '20008001', '20009001']
  RAG Top 3 (Narrative): ['20009001', '20008001', '20036001']
  Precision@3:         100.00%
  Context Precision:   100.00%

📌 Exception ID: 20039001
  Manual Top 3:        ['20029001', '20024001', '86572959']
  RAG Top 3 (Narrative): ['20024001', '20021001', '20019001']
  Precision@3:         33.33%
  Context Precision:   33.33%

📌 Exception ID: 20040001
  Manual Top 3:        ['20030001', '20033001', '20002001']
  RAG Top 3 (Narrative): ['20030001', '20025001', '20007001']
  Precision@3:         33.33%
  Context Precision:   33.33%


Metrics Summary DataFrame (Narrative Data):
  exception_id  precision_at_3  context_precision
0     20038001        1.000000           1.000000
1     20039001        0.333333           0.333333
2     20040001        0.333333           0.333333

AGGREGATE EVALUATION METRICS - NARRATIVE DATA

Total Test Exceptions: 3

Ave

## 8. Key Insights and Conclusions


In [16]:
# Create a comprehensive comparison DataFrame
comparison_summary = []

for test_exc_id in test_exception_ids:
    # Get metrics from both approaches
    raw_precision = metrics_df[metrics_df['exception_id'] == test_exc_id]['precision_at_3'].values[0]
    raw_context_prec = metrics_df[metrics_df['exception_id'] == test_exc_id]['context_precision'].values[0]
    
    narrative_precision = metrics_df_narrative[metrics_df_narrative['exception_id'] == test_exc_id]['precision_at_3'].values[0]
    narrative_context_prec = metrics_df_narrative[metrics_df_narrative['exception_id'] == test_exc_id]['context_precision'].values[0]
    
    # Calculate differences
    precision_diff = narrative_precision - raw_precision
    context_prec_diff = narrative_context_prec - raw_context_prec
    
    # Get RAG results from both
    raw_rag_ids = tuple(metrics_df[metrics_df['exception_id'] == test_exc_id]['rag_top_3'].values[0])
    narrative_rag_ids = tuple(metrics_df_narrative[metrics_df_narrative['exception_id'] == test_exc_id]['rag_top_3'].values[0])
    
    # Check if results are identical
    results_identical = raw_rag_ids == narrative_rag_ids
    
    comparison_summary.append({
        'exception_id': test_exc_id,
        'raw_precision': raw_precision,
        'narrative_precision': narrative_precision,
        'precision_delta': precision_diff,
        'raw_context_prec': raw_context_prec,
        'narrative_context_prec': narrative_context_prec,
        'context_prec_delta': context_prec_diff,
        'results_identical': results_identical,
        'raw_rag_ids': raw_rag_ids,
        'narrative_rag_ids': narrative_rag_ids
    })

comparison_df = pd.DataFrame(comparison_summary)

print("\n" + "="*90)
print("COMPARISON: RAG Performance with Raw JSON vs Narrative Data")
print("="*90)

# Detailed comparison per exception
for idx, row in comparison_df.iterrows():
    print(f"\n📊 Exception ID: {row['exception_id']}")
    print(f"\n  Precision@3:")
    print(f"    Raw JSON:       {row['raw_precision']:.2%}")
    print(f"    Narrative:      {row['narrative_precision']:.2%}")
    print(f"    Change:         {row['precision_delta']:+.2%} {'📈' if row['precision_delta'] > 0 else '📉' if row['precision_delta'] < 0 else '〰️'}")
    
    print(f"\n  Context Precision:")
    print(f"    Raw JSON:       {row['raw_context_prec']:.2%}")
    print(f"    Narrative:      {row['narrative_context_prec']:.2%}")
    print(f"    Change:         {row['context_prec_delta']:+.2%} {'📈' if row['context_prec_delta'] > 0 else '📉' if row['context_prec_delta'] < 0 else '〰️'}")
    
    print(f"\n  RAG Results:")
    print(f"    Raw JSON:       {list(row['raw_rag_ids'])}")
    print(f"    Narrative:      {list(row['narrative_rag_ids'])}")
    print(f"    Identical:      {'✓ Yes' if row['results_identical'] else '✗ No'}")

# Aggregate comparison
print("\n" + "="*90)
print("AGGREGATE COMPARISON")
print("="*90)

raw_avg_precision = metrics_df['precision_at_3'].mean()
narrative_avg_precision = metrics_df_narrative['precision_at_3'].mean()
raw_avg_context = metrics_df['context_precision'].mean()
narrative_avg_context = metrics_df_narrative['context_precision'].mean()

print(f"\n📌 PRECISION@3:")
print(f"  Raw JSON:       {raw_avg_precision:.2%}")
print(f"  Narrative:      {narrative_avg_precision:.2%}")
print(f"  Difference:     {(narrative_avg_precision - raw_avg_precision):+.2%}")

print(f"\n📌 CONTEXT PRECISION:")
print(f"  Raw JSON:       {raw_avg_context:.2%}")
print(f"  Narrative:      {narrative_avg_context:.2%}")
print(f"  Difference:     {(narrative_avg_context - raw_avg_context):+.2%}")

# Overall assessment
identical_count = comparison_df['results_identical'].sum()
print(f"\n📌 IDENTICAL RESULTS:")
print(f"  {identical_count}/3 test exceptions returned same RAG results")
print(f"  Note: Even with different representations, RAG may find the same similar exceptions")

print("\n" + "="*90)
print("KEY INSIGHTS")
print("="*90)

if narrative_avg_precision > raw_avg_precision and narrative_avg_context > raw_avg_context:
    print("\n✓ Narrative data improves both precision and context precision")
    print("  → Human-readable text helps RAG better understand exception relationships")
elif narrative_avg_precision < raw_avg_precision or narrative_avg_context < raw_avg_context:
    print("\n✗ Raw JSON data performs better on at least one metric")
    print("  → Structured JSON may contain more distinguishing features than narrative")
else:
    print("\n〰️ Performance is comparable between both approaches")
    print("  → Narrative and raw JSON representations are equally effective for RAG")

print("\n" + "="*90)



COMPARISON: RAG Performance with Raw JSON vs Narrative Data

📊 Exception ID: 20038001

  Precision@3:
    Raw JSON:       0.00%
    Narrative:      100.00%
    Change:         +100.00% 📈

  Context Precision:
    Raw JSON:       0.00%
    Narrative:      100.00%
    Change:         +100.00% 📈

  RAG Results:
    Raw JSON:       ['20040001', '20039001', '20035001']
    Narrative:      ['20009001', '20008001', '20036001']
    Identical:      ✗ No

📊 Exception ID: 20039001

  Precision@3:
    Raw JSON:       33.33%
    Narrative:      33.33%
    Change:         +0.00% 〰️

  Context Precision:
    Raw JSON:       33.33%
    Narrative:      33.33%
    Change:         +0.00% 〰️

  RAG Results:
    Raw JSON:       ['20040001', '20024001', '20021001']
    Narrative:      ['20024001', '20021001', '20019001']
    Identical:      ✗ No

📊 Exception ID: 20040001

  Precision@3:
    Raw JSON:       33.33%
    Narrative:      33.33%
    Change:         +0.00% 〰️

  Context Precision:
    Raw JSON:  